# 📖 Notebook 1: Count-Min Sketch

Before we can find the top-K most viewed videos, we need a way to **count** how many times each video has been viewed. The obvious answer is a hash map (Python dictionary) — but at YouTube scale with billions of unique videos, the counters alone are **~54 GB**, and a live Python dict costs **~270 GB** once you pay for object overhead. We measure both in the next cell.

Count-Min Sketch (CMS) is a probabilistic data structure that counts items using a **fixed amount of memory** — regardless of how many unique items you throw at it. The trade-off? It might **overcount**, but it will never undercount — the error is strictly one-sided, and we'll pin down exactly how big it can get.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why exact counting breaks down at scale
- How Count-Min Sketch uses hash functions and a 2D array to approximate counts
- How to build a CMS from scratch in Python
- The relationship between sketch size, accuracy, and memory usage
- How CMS compares to exact counting on real data from our database

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/top-k
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `topk_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import random
import mmh3  # MurmurHash3 — a fast, non-cryptographic hash function

# Database connection settings (matches docker-compose.yml)
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "topk_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker compose up -d")

## 🤔 Why Not Just Use a Dictionary?

Let's start with the obvious approach: count video views in a Python dictionary.

This works perfectly when you have a small number of items. But what happens when you have **billions** of unique video IDs?

In [ ]:
import sys

# Exact counting with a dictionary — simple and correct
exact_counts = {}

# Simulate counting views for different numbers of unique videos
print("📊 Memory Usage of Exact Counting (Python dict)")
print("=" * 55)
print(f"{'Unique Videos':>15} {'Memory':>12}  {'Bytes/entry':>11}")
print("-" * 55)

for num_videos in [100, 10_000, 1_000_000]:
    counts = {f"vid_{i:012d}": random.randint(1, 1_000_000) for i in range(num_videos)}
    memory_bytes = sys.getsizeof(counts)
    # Each entry also has key + value overhead
    total_bytes = memory_bytes + sum(
        sys.getsizeof(k) + sys.getsizeof(v) for k, v in counts.items()
    )
    if total_bytes < 1024 * 1024:
        mem_str = f"{total_bytes / 1024:.1f} KB"
    else:
        mem_str = f"{total_bytes / (1024 * 1024):.1f} MB"
    print(f"{num_videos:>15,} {mem_str:>12}  {total_bytes/num_videos:>11.0f}")

print()
# Extrapolate to YouTube scale. Two figures, because both get quoted and they
# are NOT the same number:
#   • 16 B/entry  = the information itself (8-byte id + 8-byte counter)
#   • ~80 B/entry = what a live Python dict actually costs (measured above)
raw_gb = 3_600_000_000 * 16 / (1024**3)
dict_gb = 3_600_000_000 * 80 / (1024**3)
print(f"🔮 Memory for 3.6B videos, raw (id + count) : ~{raw_gb:.0f} GB")
print(f"🔮 Memory for 3.6B videos, Python dict      : ~{dict_gb:.0f} GB")
print()
print("💡 That's more RAM than most servers have!")
print("   We need a smarter approach — enter Count-Min Sketch.")

## 🧠 How Count-Min Sketch Works

Count-Min Sketch uses a **2D array** of counters (called a "sketch") plus **multiple hash functions**.

Think of it like this analogy:

> Imagine you have 5 friends, and you ask each of them to keep a tally of video views.  
> But instead of tracking every video separately, each friend has only 1000 tally slots.  
> Each friend assigns videos to slots using their own personal rule (hash function).  
> Different friends might put the same video in different slots.
> 
> When you want to know how many views `vid_042` got, you ask all 5 friends:  
> "What's the count in the slot where you put vid_042?"  
> You take the **minimum** of all their answers — that's your best estimate.

The "min" in Count-**Min** Sketch comes from this step!

```
                    Columns (width w)
              ┌───┬───┬───┬───┬───┬───┐
  Row 0 (h₀) │ 0 │ 3 │ 0 │ 7 │ 1 │ 0 │  ← hash₀("vid_042") points here
              ├───┼───┼───┼───┼───┼───┤
  Row 1 (h₁) │ 2 │ 0 │ 5 │ 0 │ 0 │ 4 │  ← hash₁("vid_042") points here
              ├───┼───┼───┼───┼───┼───┤
  Row 2 (h₂) │ 0 │ 0 │ 0 │ 3 │ 8 │ 0 │  ← hash₂("vid_042") points here
              └───┴───┴───┴───┴───┴───┘

  estimate("vid_042") = min(7, 5, 3) = 3
```

### Key Properties
- **Never undercounts** — a counter can only go up, so the true count ≤ the estimate
- **May overcount** — other items might hash to the same slot (collision), inflating the count
- **Fixed memory** — the sketch size is fixed regardless of how many unique items you count
- **More rows + columns = more accurate** — but uses more memory

In [ ]:
class CountMinSketch:
    """
    Count-Min Sketch — a probabilistic frequency counter.
    
    Parameters:
        width:  number of columns (more = fewer collisions = more accurate)
        depth:  number of rows / hash functions (more = lower chance of ALL rows colliding)
    
    Memory usage: width × depth × 4 bytes (using 32-bit integers)
    """
    
    def __init__(self, width: int, depth: int):
        self.width = width
        self.depth = depth
        # Create a 2D array of zeros: depth rows × width columns
        self.table = [[0] * width for _ in range(depth)]
    
    def _hash(self, item: str, row: int) -> int:
        """
        Hash an item to a column index for a given row.
        We use MurmurHash3 with a different seed per row
        so each row has a different hash function.
        """
        return mmh3.hash(item, seed=row) % self.width
    
    def add(self, item: str, count: int = 1):
        """
        Record 'count' occurrences of 'item'.
        Increments one counter per row.
        """
        for row in range(self.depth):
            col = self._hash(item, row)
            self.table[row][col] += count
    
    def estimate(self, item: str) -> int:
        """
        Estimate the count of 'item'.
        Returns the MINIMUM across all rows — this is the closest
        to the true count because it's least affected by collisions.
        """
        return min(
            self.table[row][self._hash(item, row)]
            for row in range(self.depth)
        )
    
    def memory_bytes(self) -> int:
        """Approximate memory usage in bytes (4 bytes per counter)."""
        return self.width * self.depth * 4

print("✅ CountMinSketch class defined!")
print()
print("Let's test it with a simple example...")

In [ ]:
# Simple demo: count fruit
cms = CountMinSketch(width=10, depth=3)

# Add some items
cms.add("apple", 5)
cms.add("banana", 3)
cms.add("cherry", 7)
cms.add("apple", 2)  # apple now has 7 total

print("🍎 Simple CMS Demo (width=10, depth=3)")
print("=" * 40)
print(f"  apple:  estimated={cms.estimate('apple'):>3}  (true=7)")
print(f"  banana: estimated={cms.estimate('banana'):>3}  (true=3)")
print(f"  cherry: estimated={cms.estimate('cherry'):>3}  (true=7)")
print(f"  grape:  estimated={cms.estimate('grape'):>3}  (true=0)")
print()
print("💡 With only 10 columns and 3 rows, estimates are already close!")
print("   But 'grape' might show > 0 due to hash collisions.")
print()

# Show the actual table
print("📋 The sketch table (each row uses a different hash function):")
for i, row in enumerate(cms.table):
    print(f"  Row {i} (hash_{i}): {row}")

## 📊 Testing CMS on Real Video Data

Now let's use our CMS on the actual view events from our PostgreSQL database.  
We'll compare the CMS estimates against the exact counts to see how accurate it is.

In [ ]:
# Load all view events from the database
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT video_id FROM view_events ORDER BY viewed_at")
view_events = [row[0] for row in cursor.fetchall()]
conn.close()

print(f"📦 Loaded {len(view_events):,} view events from PostgreSQL")
print(f"   Unique videos: {len(set(view_events))}")
print()

# Get exact counts for comparison
exact = {}
for vid in view_events:
    exact[vid] = exact.get(vid, 0) + 1

# Show top 10 by exact count
top_exact = sorted(exact.items(), key=lambda x: x[1], reverse=True)[:10]
print("🏆 Top 10 videos by exact count:")
for rank, (vid, count) in enumerate(top_exact, 1):
    bar = "█" * (count // 50)
    print(f"  {rank:>2}. {vid}: {count:>6} views  {bar}")

In [ ]:
# Now count the same events using CMS and compare
# We'll try different sketch sizes to see the accuracy trade-off

print("📊 CMS Accuracy vs Sketch Size")
print("=" * 70)

configs = [
    (50, 3, "Tiny"),
    (100, 5, "Small"),
    (500, 5, "Medium"),
    (1000, 7, "Large"),
]

for width, depth, label in configs:
    cms = CountMinSketch(width=width, depth=depth)
    
    # Feed all view events into the sketch
    for vid in view_events:
        cms.add(vid)
    
    # Compare estimates vs exact counts
    errors = []
    for vid, true_count in exact.items():
        est = cms.estimate(vid)
        # The guarantee is ONE-SIDED: a counter only ever goes up, so the
        # minimum across rows can never fall below the true count. If this
        # ever trips, the sketch (or the hashing) is broken.
        assert est >= true_count, (
            f"{label} sketch UNDERCOUNTED {vid}: est={est} < true={true_count}"
        )
        error_pct = ((est - true_count) / true_count) * 100
        errors.append(error_pct)
    
    avg_error = sum(errors) / len(errors)
    max_error = max(errors)
    min_error = min(errors)
    assert min_error >= 0, f"{label}: negative error {min_error:.1f}% means an undercount"
    mem = cms.memory_bytes()
    
    print(f"\n  {label} ({width}×{depth} = {width*depth:,} counters, {mem:,} bytes)")
    print(f"    Avg overcount: {avg_error:>6.1f}%")
    print(f"    Max overcount: {max_error:>6.1f}%")
    
    # Show top 5 comparison
    print(f"    Top 5 comparison:")
    for vid, true_count in top_exact[:5]:
        est = cms.estimate(vid)
        diff = est - true_count
        marker = "✅" if diff == 0 else f"⚠️  +{diff}"
        print(f"      {vid}: exact={true_count:>5}, CMS={est:>5}  {marker}")

print()
print("💡 Key insight: CMS NEVER undercounts (estimates ≥ true count).")
print("   Every estimate above was asserted to be ≥ the exact count, so the")
print("   error bars are one-sided — they only ever point up, never down.")
print("   Bigger sketch = fewer collisions = more accurate estimates.")
print("   But even a tiny sketch gives useful approximations!")

## ⚖️ Memory Comparison: Exact vs CMS

Let's see how much memory we save by using CMS instead of a dictionary.  
Remember: at YouTube scale, this is the difference between needing one server and hundreds.

In [ ]:
print("💾 Memory Comparison: Exact Dictionary vs Count-Min Sketch")
print("=" * 65)
print()

# Our lab data
num_unique = len(exact)
dict_memory = sum(sys.getsizeof(k) + sys.getsizeof(v) for k, v in exact.items())
dict_memory += sys.getsizeof(exact)

cms_medium = CountMinSketch(500, 5)
cms_memory = cms_medium.memory_bytes()

print(f"  Our lab ({num_unique} unique videos):")
print(f"    Dictionary: {dict_memory:>10,} bytes ({dict_memory/1024:.1f} KB)")
print(f"    CMS 500x5:  {cms_memory:>10,} bytes ({cms_memory/1024:.1f} KB)")
print()
print("  ⚠️  At this tiny scale the dict actually WINS — CMS has a fixed")
print("     overhead (500x5 counters) that only pays off when you have")
print("     many unique keys. CMS is the right tool at LARGE scale only.")
print()

# YouTube scale extrapolation
print(f"  YouTube scale (3.6 billion unique videos):")
yt_dict_bytes = 3_600_000_000 * 80  # ~80 bytes per entry
yt_cms_bytes = 10_000_000 * 10 * 4  # 10M x 10 sketch
print(f"    Dictionary: {yt_dict_bytes / (1024**3):>10.1f} GB")
print(f"    CMS:        {yt_cms_bytes / (1024**2):>10.1f} MB")
print(f"    Savings:    {(1 - yt_cms_bytes/yt_dict_bytes)*100:.2f}%")
print()
print("🚀 CMS uses ~400 MB instead of ~268 GB — a ~700x reduction.")
print("   That's the difference between one server and a whole cluster.")


## 🌪️ A More Realistic Stream: 1 Million Events, Heavy Collisions

Our lab data is too small to see Count-Min Sketch struggle. In real systems,
you have **billions** of distinct items — and any reasonable sketch will have
real collisions. Let's simulate 1M events over 20,000 unique videos with a
**Zipf-like distribution** (a few viral videos and a very long tail, just like
real YouTube) and watch CMS accuracy degrade gracefully as we shrink the sketch.


In [ ]:
import random
random.seed(42)

NUM_UNIQUE = 20_000
NUM_EVENTS = 1_000_000

def zipf_sample(n):
    # u**2.5 biases heavily toward low ranks → a few viral items, long tail
    u = random.random()
    return int(n * (u ** 2.5))

stream = [f"vid_{zipf_sample(NUM_UNIQUE):06d}" for _ in range(NUM_EVENTS)]

exact_big = {}
for v in stream:
    exact_big[v] = exact_big.get(v, 0) + 1

print(f"Generated {NUM_EVENTS:,} events over {len(exact_big):,} unique videos")
top5 = sorted(exact_big.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 true counts:")
for v, c in top5:
    print(f"  {v}: {c:,} views")


In [ ]:
# Measure overcount error for different sketch sizes on the top-100 videos
# (the ones we actually care about for a top-K answer).

configs = [
    (500,    3, "Very tiny"),
    (2_000,  4, "Small"),
    (10_000, 5, "Medium"),
    (50_000, 5, "Large"),
]

top100_vids = [v for v, _ in sorted(exact_big.items(), key=lambda x: x[1], reverse=True)[:100]]

print(f"{'Size':<12} {'Counters':>12} {'Memory':>10} {'Avg%':>8} {'Max%':>8} {'Top-1 err%':>12}")
print("-" * 68)

for w, d, label in configs:
    cms_big = CountMinSketch(width=w, depth=d)
    for v in stream:
        cms_big.add(v)

    errs = []
    for v in top100_vids:
        true_c = exact_big[v]
        est = cms_big.estimate(v)
        assert est >= true_c, f"{label}: undercounted {v} ({est} < {true_c})"
        errs.append((est - true_c) / true_c * 100)

    top1_err = errs[0]
    avg = sum(errs) / len(errs)
    mx = max(errs)
    mem_kb = cms_big.memory_bytes() / 1024
    print(f"{label:<12} {w*d:>12,} {mem_kb:>8.0f}KB {avg:>7.2f}% {mx:>7.2f}% {top1_err:>11.2f}%")

print()
print("💡 Three things to notice:")
print("   1. Tiny sketches overcount badly — every collision adds noise.")
print("   2. The #1 video's % error is far BELOW the average, because collision")
print("      noise is roughly a constant number of extra counts and its true")
print("      count is huge. (It is usually the single lowest, but not always —")
print("      the guarantee is about the absolute overcount, not the ranking of")
print("      percentages.) Either way, CMS is naturally good at top-K.")
print("   3. Accuracy is cheap here: the 'Medium' sketch is ~195 KB for ~2%")
print("      average error, against ~1.9 MB for a plain dict of 20,000 keys.")
print("      The 'Large' sketch buys exactness but costs ~977 KB — nearly the")
print("      dict — which is the point where you should stop growing it.")

# Guard the claim in note 2: the #1 item's error must beat the top-100 average.
assert top1_err <= avg, (
    f"the most popular video's error ({top1_err:.2f}%) exceeded the top-100 "
    f"average ({avg:.2f}%) — the 'popular items are most accurate' claim fails"
)
print()
print(f"✅ Top-1 error {top1_err:.2f}% ≤ top-100 average {avg:.2f}% (last config)")


## 📐 Sizing the Sketch: ε and δ

So far we have picked `width` and `depth` by feel. There is an actual formula,
and it is the thing interviewers ask about.

Pick two knobs:

| Knob | Meaning |
|---|---|
| **ε** (epsilon) | how much overcount you tolerate, as a fraction of the **total stream size** |
| **δ** (delta) | the probability that an estimate is allowed to blow past that tolerance |

Then size the sketch:

$$w = \left\lceil \frac{e}{\varepsilon} \right\rceil \qquad d = \left\lceil \ln\frac{1}{\delta} \right\rceil$$

and the guarantee you buy is:

$$\widehat{f}(x) \;\ge\; f(x) \qquad\text{always}$$
$$\Pr\!\left[\, \widehat{f}(x) \;>\; f(x) + \varepsilon N \,\right] \;\le\; \delta$$

where $f(x)$ is the true count, $\widehat f(x)$ the estimate, and $N$ the total
number of events counted.

Three things people get wrong about this, all of them worth saying out loud:

1. **The bound is one-sided.** The first line is not probabilistic — it holds
   for every item, every time. There is no `± error`; the estimate is never
   too low. Anyone quoting a symmetric error bar for CMS has it wrong.
2. **The error scales with N, not with the item's own count.** A tolerance of
   ε = 0.1% on a 1M-event stream means "up to 1,000 extra views" — trivial for
   a video with 19,000 views, fatal for one with 3.
3. **Width buys accuracy, depth buys confidence.** Halving ε doubles the width
   linearly; shrinking δ by 10x costs only ~2.3 more rows, because depth sits
   inside a logarithm. Depth is cheap; width is what you pay for.

Let's size a sketch from (ε, δ) and check the guarantee against real numbers.


In [ ]:
import math

def size_for(epsilon: float, delta: float) -> tuple:
    """Standard Cormode-Muthukrishnan sizing: w = ⌈e/ε⌉, d = ⌈ln(1/δ)⌉."""
    return math.ceil(math.e / epsilon), math.ceil(math.log(1 / delta))

N = len(stream)  # total events counted — the bound is relative to THIS
print(f"Stream: {N:,} events over {len(exact_big):,} unique videos")
print()
print(f"{'ε':>8} {'δ':>7} {'width':>8} {'depth':>6} {'memory':>10} "
      f"{'ε·N budget':>11} {'worst over':>11} {'over budget':>12}")
print("-" * 82)

for epsilon, delta in [(0.01, 0.01), (0.001, 0.01), (0.0001, 0.001)]:
    w, d = size_for(epsilon, delta)
    sized = CountMinSketch(width=w, depth=d)
    for v in stream:
        sized.add(v)

    budget = epsilon * N
    overcounts = []
    for vid, true_c in exact_big.items():
        est = sized.estimate(vid)
        # Line 1 of the guarantee — deterministic, no probability involved.
        assert est >= true_c, f"undercount on {vid}: {est} < {true_c}"
        overcounts.append(est - true_c)

    over_budget = sum(1 for o in overcounts if o > budget)
    observed_delta = over_budget / len(overcounts)

    print(f"{epsilon:>8} {delta:>7} {w:>8,} {d:>6} "
          f"{sized.memory_bytes()/1024:>8.0f}KB {budget:>11,.0f} "
          f"{max(overcounts):>11,} {over_budget:>6} ({observed_delta:>.4f})")

    # Line 2 of the guarantee — the failure rate must stay under δ.
    assert observed_delta <= delta, (
        f"ε={epsilon}, δ={delta}: {over_budget}/{len(overcounts)} items exceeded "
        f"the ε·N budget of {budget:,.0f} (rate {observed_delta:.4f} > δ={delta})"
    )

print()
print("💡 Read the two right-hand columns together:")
print("   • 'worst over' is the largest absolute overcount seen ANYWHERE.")
print("     It stays comfortably inside the ε·N budget at every setting.")
print("   • 'over budget' is how many of the 20,000 videos broke the budget.")
print("     Zero, every time — the bound is a worst case, not a target.")
print()
print("⚠️  Note what ε·N means in practice. At ε=0.01 on a 1M-event stream the")
print("    budget is 10,000 extra views — bigger than the true count of every")
print("    video except the #1. The bound is only useful for HEAVY hitters;")
print("    for the long tail, CMS tells you almost nothing. That is fine: the")
print("    long tail is exactly what we are not asking about in a top-K query.")

## 🔬 Visualizing Hash Collisions

The only source of error in CMS is **hash collisions** — when two different items hash to the same slot.  
Let's visualize how items spread across the sketch and where collisions happen.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# Build a small CMS so we can visualize it
vis_cms = CountMinSketch(width=20, depth=4)
for vid in view_events:
    vis_cms.add(vid)

# Plot the sketch as a heatmap
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(vis_cms.table, aspect='auto', cmap='YlOrRd')
ax.set_xlabel('Column (hash bucket)')
ax.set_ylabel('Row (hash function)')
ax.set_title('Count-Min Sketch Heatmap (20×4) — Darker = More Counts')
ax.set_yticks(range(4))
ax.set_yticklabels([f'hash_{i}' for i in range(4)])
plt.colorbar(im, label='Count')
plt.tight_layout()
plt.show()

print("💡 Hot spots (dark cells) indicate hash collisions.")
print("   Multiple videos are mapped to the same slot, inflating the count.")
print("   Increasing width spreads items more evenly → fewer collisions.")

In [ ]:
# Visualize the error distribution
cms_test = CountMinSketch(width=200, depth=5)
for vid in view_events:
    cms_test.add(vid)

errors = []
labels = []
for vid, true_count in sorted(exact.items(), key=lambda x: x[1], reverse=True):
    est = cms_test.estimate(vid)
    error_pct = ((est - true_count) / true_count) * 100
    errors.append(error_pct)
    labels.append(vid)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['green' if e < 5 else 'orange' if e < 20 else 'red' for e in errors]
ax.bar(range(len(errors)), errors, color=colors)
ax.set_xlabel('Videos (sorted by true view count, most popular on left)')
ax.set_ylabel('Overcount Error (%)')
ax.set_title('CMS Error by Video (200×5 sketch)')
ax.axhline(y=0, color='black', linewidth=0.5)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', label='< 5% error'),
    Patch(facecolor='orange', label='5-20% error'),
    Patch(facecolor='red', label='> 20% error'),
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

print("💡 Popular videos (left) tend to have lower % error.")
print("   Why? Because the collision noise is small relative to their large true count.")
print("   This is great for top-K: the videos we care about are the most accurate!")

## 🔗 CMS in Redis

Redis has **built-in Count-Min Sketch** support! In a real system, you'd use Redis CMS  
instead of building your own. Let's try it with our video data.

In [ ]:
r = get_redis_client()

# Create a Count-Min Sketch in Redis
# CMS.INITBYPROB key error_rate probability
#   error_rate: how much overcount we tolerate (0.01 = 1%)
#   probability: chance of exceeding error_rate (0.01 = 1%)
try:
    r.execute_command('CMS.INITBYPROB', 'notebook1:video_cms', 0.01, 0.01)
    print("✅ Created Redis CMS 'notebook1:video_cms'")
except redis.exceptions.ResponseError as e:
    if 'exists' in str(e).lower():
        print("ℹ️  CMS already exists, reusing it")
    else:
        print(f"⚠️  Redis CMS not available: {e}")
        print("   This requires the RedisBloom module.")
        print("   Our Python CMS above works the same way!")

In [ ]:
# Feed view events into the Redis CMS
try:
    # Batch the increments for efficiency
    # CMS.INCRBY key item count [item count ...]
    batch = {}
    for vid in view_events:
        batch[vid] = batch.get(vid, 0) + 1
    
    args = []
    for vid, count in batch.items():
        args.extend([vid, count])
    
    r.execute_command('CMS.INCRBY', 'notebook1:video_cms', *args)
    print(f"✅ Added {len(view_events):,} events to Redis CMS")
    
    # Query the Redis CMS
    print("\n🏆 Redis CMS estimates vs exact counts (top 10):")
    print(f"   {'Video':<12} {'Exact':>7} {'Redis CMS':>10} {'Error':>7}")
    print("   " + "-" * 40)
    
    for vid, true_count in top_exact:
        result = r.execute_command('CMS.QUERY', 'notebook1:video_cms', vid)
        est = result[0]
        err = est - true_count
        marker = "✅" if err == 0 else f"+{err}"
        print(f"   {vid:<12} {true_count:>7} {est:>10} {marker:>7}")

except redis.exceptions.ResponseError:
    print("⚠️  Redis CMS commands not available (needs RedisBloom module).")
    print("   No worries — our Python CMS above demonstrates the same concepts!")

## 🧹 Cleanup

In [ ]:
# Clean up Redis keys
r = get_redis_client()
keys = r.keys("notebook1:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

## 🎯 Bonus: Redis `TOPK` — a Purpose-Built Top-K Data Structure

Redis Stack ships with a **TOPK** data structure (based on the HeavyKeeper
algorithm) that combines a sketch AND a min-heap in a single server-side
primitive — exactly the pattern we'll build by hand in Notebook 2. You tell it
how many top items you want and it maintains them as you stream data in.

Think of it as "CMS + heap, handled by Redis for you."


In [ ]:
r = get_redis_client()

# TOPK.RESERVE key topk width depth decay
#   topk:  how many top items to track
#   width, depth: sketch dimensions (accuracy knobs)
#   decay: probability of demoting an item on collision (0.9 is standard)
try:
    r.delete("notebook1:video_topk")
    r.execute_command("TOPK.RESERVE", "notebook1:video_topk", 10, 200, 5, 0.9)

    # Stream events in with TOPK.ADD. We batch ~1000 per call.
    chunk = []
    for v in view_events:
        chunk.append(v)
        if len(chunk) >= 1000:
            r.execute_command("TOPK.ADD", "notebook1:video_topk", *chunk)
            chunk = []
    if chunk:
        r.execute_command("TOPK.ADD", "notebook1:video_topk", *chunk)

    # TOPK.LIST key [WITHCOUNT] — read the current top K
    result = r.execute_command("TOPK.LIST", "notebook1:video_topk", "WITHCOUNT")
    print("🏆 Redis TOPK.LIST result (top 10 from the stream):")
    for j in range(0, len(result), 2):
        vid = result[j]
        count = int(result[j+1])
        true_count = exact.get(vid, 0)
        marker = "✅" if count == true_count else f"~{count} (true={true_count})"
        print(f"  {vid}: {marker}")
    print()
    print("💡 TOPK is one Redis command away. In production you'd use this")
    print("   instead of rolling your own CMS + heap for simple cases.")

except redis.exceptions.ResponseError as e:
    print(f"⚠️  TOPK commands not available on this Redis: {e}")
    print("   (Requires Redis Stack / RedisBloom module.)")


## 📚 Summary

### Key Takeaways

1. **Exact counting doesn't scale** — at billions of unique items, a hash map needs 100+ GB of RAM
2. **Count-Min Sketch trades accuracy for memory** — it uses a fixed-size 2D array regardless of item count
3. **CMS never undercounts** — the error is strictly **one-sided**. There is no `± error bar`;
   `estimate ≥ true` holds for every item, always. Anyone quoting a symmetric error bar for
   CMS has it wrong.
4. **Size it from ε and δ** — `w = ⌈e/ε⌉`, `d = ⌈ln(1/δ)⌉` buys you
   `P[estimate > true + ε·N] ≤ δ`. Width buys accuracy (linear), depth buys confidence
   (logarithmic — so depth is cheap and width is what you pay for).
5. **The tolerance scales with N, not with the item** — `ε·N` is an absolute number of extra
   counts. It is negligible for a heavy hitter and useless for the long tail, which is exactly
   why CMS suits top-K and not "how many views did this obscure video get?"
6. **Popular items are most accurate** — collision noise is roughly constant, so it shrinks as a
   percentage of a large true count

### CMS Cheat Sheet

| Operation | Time Complexity | Description |
|-----------|----------------|-------------|
| `add(item, count)` | O(depth) | Increment `depth` counters |
| `estimate(item)` | O(depth) | Read `depth` counters, return minimum |
| Memory | O(width × depth) | Fixed, independent of number of unique items |
| Merge two sketches | O(width × depth) | Element-wise add — same dimensions and seeds required |

### Next Up

CMS tells us *approximately how many views* a video has. But we still need to find the **top K** videos efficiently.  
In **Notebook 2**, we'll combine CMS with a **min-heap** to maintain a running top-K from a stream of events.